# 💬 Chat with NVIDIA Nemotron-3-Nano-30B via vLLM

This notebook sets up a fast, interactive multi-turn chat session with the **NVIDIA-Nemotron-3-Nano-30B** base reasoning model using **vLLM**.

It automatically handles:
1. **Offline Wheels Installation** (if run offline in Kaggle).
2. **Blackwell GPU Compatibility Patches** (SM 12.0 to SM 9.0 spoofing).
3. **Multi-Turn Chat History Tracking** with proper chat templates.

In [ ]:
import os
import glob
import subprocess

# Install all local packages offline from the ml-wheels directory
paths_to_check = [
    "/kaggle/input/datasets/ckskaggle/vllm-offline-06-26/vllm-offline/",
    "/kaggle/input/vllm-offline-06-26/vllm-offline/",
    "/kaggle/input/vllm-offline/",
    "/kaggle/input/datasets/ckskaggle/ml-wheels/"
]
wheels_dir = None
for p in paths_to_check:
    if os.path.exists(p):
        wheels_dir = p
        break
if wheels_dir is None:
    wheels_dir = "/kaggle/input/datasets/ckskaggle/ml-wheels/"

if os.path.exists(wheels_dir):
    print(f"Found offline wheels directory: {wheels_dir}")
    wheel_files = glob.glob(os.path.join(wheels_dir, "*.whl"))
    if wheel_files:
        print(f"Found {len(wheel_files)} wheels. Installing offline...")
        cmd = f"pip install --no-index --find-links={wheels_dir} --target=/kaggle/working --ignore-installed " + " ".join(wheel_files)
        print(f"Running: {cmd}")
        try:
            subprocess.run(cmd, shell=True, check=True)
            print("All offline wheels installed successfully!")
        except Exception as e:
            print(f"Offline installation failed: {e}")
    else:
        print("No wheel (.whl) files found in the directory.")
else:
    print("Offline wheels directory not found. Assuming environment is already set up.")

In [ ]:
import os
import shutil
import stat
import glob
import site
import sys
import torch

# --- 1. CRITICAL NVIDIA / TRITON PTXAS PERMISSION PATCH ---
print("Configuring Triton binary overrides...")
candidates = glob.glob("/kaggle/usr/lib/notebooks/*/nvidia*utility*script/triton/backends/nvidia/bin/ptxas-blackwell")
if candidates:
    src = candidates[0]
    dst_dir = "/tmp/triton-bin"
    os.makedirs(dst_dir, exist_ok=True)
    dst = f"{dst_dir}/ptxas-blackwell"
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    os.environ["TRITON_PTXAS_PATH"] = dst
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst
    os.environ["PATH"] = f"{dst_dir}:" + os.environ["PATH"]
    os.environ["CUDA_MODULE_LOADING"] = "LAZY"
    os.environ["TRITON_DISABLE_AUTOTUNE"] = "1"
    os.environ["USE_TRITON"] = "0"
    os.environ["PTXAS_PATH"] = dst
    print(f"PTXAS override successful: {dst}")
else:
    print("ptxas-blackwell not found in utility scripts. Assuming other architecture or local run.")

# --- 2. CRITICAL CACHING DIRECTORY REDIRECTS ---
print("Configuring caching directories to /tmp...")
os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["TORCH_EXTENSIONS_DIR"] = "/tmp/torch_extensions"
os.environ["TRITON_CACHE_DIR"] = "/tmp/triton_cache"

# --- 3. NVIDIA CUTLASS PATH CONFIGURATION ---
search_path = "/kaggle/usr/lib/notebooks/*/nvidia*utility*script/nvidia_cutlass_dsl/python_packages/"
candidates = glob.glob(search_path)
if candidates:
    cutlass_pkg_path = candidates[0]
    site.addsitedir(cutlass_pkg_path)
    print(f"NVIDIA Cutlass DSL package path loaded from: {cutlass_pkg_path}")

# Add working directory to Python path
sys.path.insert(0, "/kaggle/working")

# --- 4. CRITICAL BLACKWELL SM 12.0 COMPATIBILITY SPOOFING ---
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] == 12:
    print("Detected Blackwell SM 12.0 GPU. Applying compatibility spoofing to SM 9.0 (Hopper)...")
    original_get_device_properties = torch.cuda.get_device_properties
    class MockDeviceProperties:
        def __init__(self, props):
            self._props = props
            self.major = 9
            self.minor = 0
        def __getattr__(self, name):
            return getattr(self._props, name)
    torch.cuda.get_device_capability = lambda device=None: (9, 0)
    torch.cuda.get_device_properties = lambda device=None: MockDeviceProperties(original_get_device_properties(device))

In [ ]:
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
import kagglehub

# Determine model path
candidates = [
    "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
]
MODEL_PATH = None
for c in candidates:
    if os.path.exists(c):
        MODEL_PATH = c
        break

if MODEL_PATH is None:
    print("Local paths not found. Downloading via kagglehub...")
    try:
        MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    except Exception as e:
        print(f"Kagglehub download failed: {e}. Defaulting to first candidate.")
        MODEL_PATH = candidates[0]

print(f"Using model path: {MODEL_PATH}")

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

print("Initializing vLLM Engine...")
llm = LLM(
    model=MODEL_PATH,
    tensor_parallel_size=1,
    trust_remote_code=True,
    max_model_len=8192,
    gpu_memory_utilization=0.85,
    enable_lora=False,  # Base model chat only
)
print("vLLM Engine initialized successfully.")

In [ ]:
# Configure sampling parameters
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=2048
)

print("--- Chat Session Initialized ---")
print("Type 'exit', 'quit', or 'clear' to manage the chat.\n")

conversation_history = []

def chat():
    global conversation_history
    while True:
        try:
            user_input = input("You: ").strip()
        except KeyboardInterrupt:
            print("\nSession ended by user.")
            break
            
        if not user_input:
            continue
            
        if user_input.lower() in ["exit", "quit"]:
            print("Goodbye!")
            break
            
        if user_input.lower() == "clear":
            conversation_history = []
            print("\nConversation history cleared!\n")
            continue
            
        # Append user message
        conversation_history.append({"role": "user", "content": user_input})
        
        # Format conversation using the tokenizer's chat template
        prompt = tokenizer.apply_chat_template(
            conversation_history, 
            tokenize=False, 
            add_generation_prompt=True
        )
        
        # Generate model response
        print("\nNemotron is thinking...")
        outputs = llm.generate([prompt], sampling_params=sampling_params)
        
        # Parse output
        response_text = outputs[0].outputs[0].text
        
        # Add assistant response to history
        conversation_history.append({"role": "assistant", "content": response_text})
        
        # Print response nicely
        print("\n" + "="*50)
        print(f"Nemotron:\n{response_text}")
        print("="*50 + "\n")

# Start chat
chat()